# 1. BaseModel

- 维护字典dict与json数据的包装器：
    - 字段： Field（descriptrion="",title="",default=""）

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """电影的详细描述"""
    title: str = Field("Gone With Wind", description="电影名")
    year: int = Field(default=2, description="电影发布的年份")
    director: str = Field("张导", description="电影拍摄的导演", title="导演")
    rating: float = Field(8.5, description="电影的评分")

print(Movie().model_dump())     # dict类型。字段需要有值
print(Movie().model_dump_json())  # 字符串类型。
print(Movie().model_json_schema())  # 字符串类型。

<bound method BaseModel.model_dump of Movie(title='功夫无双', year=2, director='张艺谋')>
<bound method BaseModel.model_dump_json of Movie(title='功夫无双', year=2, director='张艺谋')>


In [11]:
from pydantic import BaseModel,Field
from langchain.chat_models import init_chat_model

class Movie(BaseModel):
    title:str      = Field("功夫无双",description="电影片名")
    year: str      = Field("2025/05",description="电影发布的年月")
    director:str   = Field("张艺谋",description="电影的导演")

model = init_chat_model("ollama:qwen3.5:4b")
print(type(model))
model_structured = model.with_structured_output(Movie)
print(model_structured)
response = model_structured.invoke("帮我提供电影《满江红》的信息")
print(response)


<class 'langchain_ollama.chat_models.ChatOllama'>
first=RunnableBinding(bound=ChatOllama(model='qwen3.5:4b'), kwargs={'format': {'properties': {'title': {'default': '功夫无双', 'description': '电影片名', 'title': 'Title', 'type': 'string'}, 'year': {'default': '2025/05', 'description': '电影发布的年月', 'title': 'Year', 'type': 'string'}, 'director': {'default': '张艺谋', 'description': '电影的导演', 'title': 'Director', 'type': 'string'}}, 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[]) middle=[] last=PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)


KeyboardInterrupt: 

In [6]:

from typing_extensions import TypedDict, Annotated
import json

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]



json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

KeyboardInterrupt: 

- 代码说明：
     - method 指定机构画的格式：
        - 供应商
        - 定制
        
    

# 1.2 结构输出的应用

# 1.3 智能体的结构化输出

- 在model的机构化输出的基础上，agent提供了两个

- 代码说明：
    - 

In [ ]:
# 1. 导入必要的库
from typing import List, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# 2. 定义嵌套的 Pydantic 模型
class Address(BaseModel):
    """地址信息"""
    city: str = Field(description="所在城市")
    street: Optional[str] = Field(default=None, description="街道名称")

class Education(BaseModel):
    """教育经历"""
    degree: str = Field(description="获得的学位")
    major: str = Field(description="专业名称")
    graduation_year: int = Field(description="毕业年份")

class Resume(BaseModel):
    """简历信息 - 主模型（嵌套结构）"""
    name: str = Field(description="候选人的姓名")
    age: int = Field(description="候选人的年龄")
    address: Address = Field(description="候选人的居住地址")  # 嵌套模型
    education_list: List[Education] = Field(description="候选人的教育经历列表")  # 列表嵌套

# 3. 初始化模型（选择一个不支持原生结构化输出的模型来展示 ToolStrategy）
# 注意：使用支持工具调用的模型即可，ToolStrategy 会自动生效
model = init_chat_model("ollama:qwen3.5:4b")  # Qwen支持ToolStrategy

# 4. 显式使用 ToolStrategy 包装 Schema
# agent = create_agent(
#     model=model,
#     response_format=ToolStrategy(Resume),  # 关键：显式使用 ToolStrategy
#     system_prompt="你是一个专业的信息提取助手，请准确提取用户提供的简历信息。"
# )

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Resume,
        tool_message_content="简历信息提取完成，数据已保存！",  # 自定义成功提示：自定义结构化输出生成后记录在对话历史中的提示信息
        handle_errors=True  # 自动处理错误并重试
    ),
    system_prompt="你是一个专业的信息提取助手，请准确提取用户提供的简历信息。"
)

# 5. 准备输入文本（非结构化数据）
text = """
我叫杨强，今年55岁。我住在南京市双龙大道60号陆军工程大学双龙营区。
教育背景：
1989年至1993年，就读于西南师范大学，读数学教育，获得学士学位。
1993年至1996年，就读于西南师范大学，读基础数学，获得硕士学位。
"""

# 6. 调用 Agent
result = agent.invoke({
    "messages": [{"role": "user", "content": f"请从以下文本中提取简历信息：{text}"}]
})

# 7. 获取结构化输出
resume: Resume = result["structured_response"]

# 8. 验证并访问结果
print(f"姓名: {resume.name}")
print(f"年龄: {resume.age}")
print(f"地址: {resume.address.city} {resume.address.street or ''}")
print("\n教育经历:")
for edu in resume.education_list:
    print(f"  - {edu.graduation_year}年: {edu.degree} (专业: {edu.major})")

# 也可以转为字典或 JSON 格式
print("\n原始 JSON 格式:")
print(resume.model_dump_json(indent=2))